# 🌐 Mode Operasi Block Cipher & Eksperimen Lanjutan
### Mata Kuliah: MA4151 - Kriptografi
**Implementasi Murni Python (Pure Python) Tanpa Library Kriptografi Eksternal**

---

## 🎯 Tujuan Pembelajaran
1. Memahami dan mengimplementasikan skema **Padding PKCS#7** dari nol.
2. Mengimplementasikan 3 Mode Operasi Block Cipher Utama:
   - **Electronic Codebook (ECB)** & Analisis Kelemahan Pola Visual
   - **Cipher Block Chaining (CBC)** dengan Vektor Inisialisasi ($IV$)
   - **Counter Mode (CTR)** (Mengubah Block Cipher menjadi Stream Cipher)
3. Melakukan **Uji Ketahanan Error (Error Propagation)** pada ciphertext yang terdistorsi.
4. Membandingkan secara langsung performa, ruang kunci, dan difusi antara **DES vs AES**.


---
## 1. Implementasi Skema Padding PKCS#7 dari Nol
Block cipher hanya dapat mengenkripsi data dengan ukuran tepat kelipatan ukuran blok ($B = 16$ byte untuk AES, $B = 8$ byte untuk DES).


In [ ]:
def pkcs7_pad(data: bytes, block_size: int = 16) -> bytes:
    """
    Menambahkan padding PKCS#7 ke data bytes.
    Nilai setiap byte padding adalah jumlah byte yang ditambahkan.
    """
    pad_len = block_size - (len(data) % block_size)
    padding = bytes([pad_len] * pad_len)
    return data + padding

def pkcs7_unpad(padded_data: bytes, block_size: int = 16) -> bytes:
    """
    Membuang dan memvalidasi padding PKCS#7 dari data bytes.
    """
    if len(padded_data) == 0 or len(padded_data) % block_size != 0:
        raise ValueError("Panjang data tidak valid untuk unpadding.")
    
    pad_len = padded_data[-1]
    if pad_len < 1 or pad_len > block_size:
        raise ValueError("Nilai padding di luar rentang yang valid.")
    
    # Verifikasi integritas byte padding
    if padded_data[-pad_len:] != bytes([pad_len] * pad_len):
        raise ValueError("Corrupted padding detected!")
        
    return padded_data[:-pad_len]

# Uji Coba PKCS#7
pesan = b"Rahasia MA4151!"
padded = pkcs7_pad(pesan, block_size=16)
unpadded = pkcs7_unpad(padded, block_size=16)

print(f"Pesan Asli       ({len(pesan):2d} bytes): {pesan}")
print(f"Setelah Padding ({len(padded):2d} bytes): {padded.hex()}")
print(f"Setelah Unpad   ({len(unpadded):2d} bytes): {unpadded.decode()}")
assert unpadded == pesan


---
## 2. Pemuatan Modul AES Core dari Nol


In [ ]:
# Salin modul AES Core dari Notebook 02 agar mandiri
def gf_mult(a, b):
    res = 0
    for _ in range(8):
        if b & 1: res ^= a
        hi = a & 0x80
        a = (a << 1) & 0xFF
        if hi: a ^= 0x1B
        b >>= 1
    return res

def gf_inv(a):
    if a == 0: return 0
    res, base, exp = 1, a, 254
    while exp > 0:
        if exp & 1: res = gf_mult(res, base)
        base = gf_mult(base, base)
        exp >>= 1
    return res

# Bangkitkan S-Box
S_BOX = [0] * 256
INV_S_BOX = [0] * 256
for i in range(256):
    inv = gf_inv(i)
    s = inv
    for shift in (1, 2, 3, 4):
        s ^= ((inv << shift) | (inv >> (8 - shift))) & 0xFF
    s ^= 0x63
    S_BOX[i] = s
    INV_S_BOX[s] = i

RCON = [0x00, 0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]

def aes_key_expansion(key_bytes):
    w = [list(key_bytes[4*i : 4*i+4]) for i in range(4)]
    for i in range(4, 44):
        temp = list(w[i-1])
        if i % 4 == 0:
            temp = [S_BOX[b] for b in (temp[1:] + temp[:1])]
            temp[0] ^= RCON[i // 4]
        w.append([w[i-4][j] ^ temp[j] for j in range(4)])
    round_keys = []
    for r in range(11):
        round_keys.append([[w[r*4 + col][row] for col in range(4)] for row in range(4)])
    return round_keys

def aes_encrypt_block(block_bytes, round_keys):
    state = [[block_bytes[c*4 + r] for c in range(4)] for r in range(4)]
    # Initial ARK
    for r in range(4):
        for c in range(4): state[r][c] ^= round_keys[0][r][c]
    # 9 Rounds
    for rnd in range(1, 10):
        # SubBytes
        for r in range(4):
            for c in range(4): state[r][c] = S_BOX[state[r][c]]
        # ShiftRows
        state[1] = state[1][1:] + state[1][:1]
        state[2] = state[2][2:] + state[2][:2]
        state[3] = state[3][3:] + state[3][:3]
        # MixColumns
        for c in range(4):
            s0, s1, s2, s3 = state[0][c], state[1][c], state[2][c], state[3][c]
            state[0][c] = gf_mult(0x02, s0) ^ gf_mult(0x03, s1) ^ s2 ^ s3
            state[1][c] = s0 ^ gf_mult(0x02, s1) ^ gf_mult(0x03, s2) ^ s3
            state[2][c] = s0 ^ s1 ^ gf_mult(0x02, s2) ^ gf_mult(0x03, s3)
            state[3][c] = gf_mult(0x03, s0) ^ s1 ^ s2 ^ gf_mult(0x02, s3)
        # AddRoundKey
        for r in range(4):
            for c in range(4): state[r][c] ^= round_keys[rnd][r][c]
    # Final Round
    for r in range(4):
        for c in range(4): state[r][c] = S_BOX[state[r][c]]
    state[1] = state[1][1:] + state[1][:1]
    state[2] = state[2][2:] + state[2][:2]
    state[3] = state[3][3:] + state[3][:3]
    for r in range(4):
        for c in range(4): state[r][c] ^= round_keys[10][r][c]
    return bytes(state[r][c] for c in range(4) for r in range(4))

def aes_decrypt_block(block_bytes, round_keys):
    state = [[block_bytes[c*4 + r] for c in range(4)] for r in range(4)]
    for r in range(4):
        for c in range(4): state[r][c] ^= round_keys[10][r][c]
    for rnd in range(9, 0, -1):
        state[1] = state[1][3:] + state[1][:3]
        state[2] = state[2][2:] + state[2][:2]
        state[3] = state[3][1:] + state[3][:1]
        for r in range(4):
            for c in range(4): state[r][c] = INV_S_BOX[state[r][c]]
        for r in range(4):
            for c in range(4): state[r][c] ^= round_keys[rnd][r][c]
        for c in range(4):
            s0, s1, s2, s3 = state[0][c], state[1][c], state[2][c], state[3][c]
            state[0][c] = gf_mult(0x0E, s0) ^ gf_mult(0x0B, s1) ^ gf_mult(0x0D, s2) ^ gf_mult(0x09, s3)
            state[1][c] = gf_mult(0x09, s0) ^ gf_mult(0x0E, s1) ^ gf_mult(0x0B, s2) ^ gf_mult(0x0D, s3)
            state[2][c] = gf_mult(0x0D, s0) ^ gf_mult(0x09, s1) ^ gf_mult(0x0E, s2) ^ gf_mult(0x0B, s3)
            state[3][c] = gf_mult(0x0B, s0) ^ gf_mult(0x0D, s1) ^ gf_mult(0x09, s2) ^ gf_mult(0x0E, s3)
    state[1] = state[1][3:] + state[1][:3]
    state[2] = state[2][2:] + state[2][:2]
    state[3] = state[3][1:] + state[3][:1]
    for r in range(4):
        for c in range(4): state[r][c] = INV_S_BOX[state[r][c]]
    for r in range(4):
        for c in range(4): state[r][c] ^= round_keys[0][r][c]
    return bytes(state[r][c] for c in range(4) for r in range(4))

print("Modul inti AES siap digunakan untuk mode operasi!")


---
## 3. Implementasi Tiga Mode Operasi: ECB, CBC, dan CTR


In [ ]:
import os

# Helper XOR antara dua bytes
def xor_bytes(b1: bytes, b2: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(b1, b2))

# ==========================================
# 1. Electronic Codebook (ECB) Mode
# ==========================================
def encrypt_ecb(plaintext: bytes, key: bytes) -> bytes:
    round_keys = aes_key_expansion(key)
    padded = pkcs7_pad(plaintext, 16)
    ciphertext = bytearray()
    for i in range(0, len(padded), 16):
        block = padded[i : i + 16]
        ciphertext.extend(aes_encrypt_block(block, round_keys))
    return bytes(ciphertext)

def decrypt_ecb(ciphertext: bytes, key: bytes) -> bytes:
    round_keys = aes_key_expansion(key)
    decrypted_padded = bytearray()
    for i in range(0, len(ciphertext), 16):
        block = ciphertext[i : i + 16]
        decrypted_padded.extend(aes_decrypt_block(block, round_keys))
    return pkcs7_unpad(bytes(decrypted_padded), 16)

# ==========================================
# 2. Cipher Block Chaining (CBC) Mode
# ==========================================
def encrypt_cbc(plaintext: bytes, key: bytes, iv: bytes = None) -> tuple[bytes, bytes]:
    if iv is None:
        iv = os.urandom(16) # Bangkitkan IV acak 16-byte
    assert len(iv) == 16
    round_keys = aes_key_expansion(key)
    padded = pkcs7_pad(plaintext, 16)
    
    ciphertext = bytearray()
    prev_cipher_block = iv
    for i in range(0, len(padded), 16):
        block = padded[i : i + 16]
        xored_block = xor_bytes(block, prev_cipher_block)
        cipher_block = aes_encrypt_block(xored_block, round_keys)
        ciphertext.extend(cipher_block)
        prev_cipher_block = cipher_block
        
    return bytes(ciphertext), iv

def decrypt_cbc(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    round_keys = aes_key_expansion(key)
    decrypted_padded = bytearray()
    prev_cipher_block = iv
    for i in range(0, len(ciphertext), 16):
        cipher_block = ciphertext[i : i + 16]
        dec_block = aes_decrypt_block(cipher_block, round_keys)
        plain_block = xor_bytes(dec_block, prev_cipher_block)
        decrypted_padded.extend(plain_block)
        prev_cipher_block = cipher_block
    return pkcs7_unpad(bytes(decrypted_padded), 16)

# ==========================================
# 3. Counter (CTR) Mode
# ==========================================
def encrypt_ctr(plaintext: bytes, key: bytes, nonce: bytes = None) -> tuple[bytes, bytes]:
    if nonce is None:
        nonce = os.urandom(8) # Nonce 8-byte
    assert len(nonce) == 8
    round_keys = aes_key_expansion(key)
    
    ciphertext = bytearray()
    for counter in range(0, (len(plaintext) + 15) // 16):
        # Bangkitkan Counter Block 16-byte: Nonce (8 bytes) || Counter (8 bytes)
        counter_block = nonce + counter.to_bytes(8, byteorder='big')
        keystream_block = aes_encrypt_block(counter_block, round_keys)
        
        chunk = plaintext[counter * 16 : (counter + 1) * 16]
        ciphertext.extend(xor_bytes(chunk, keystream_block[:len(chunk)]))
        
    return bytes(ciphertext), nonce

def decrypt_ctr(ciphertext: bytes, key: bytes, nonce: bytes) -> bytes:
    # Pada mode CTR, fungsi dekripsi persis sama dengan fungsi enkripsi!
    decrypted, _ = encrypt_ctr(ciphertext, key, nonce)
    return decrypted


---
## 4. Eksperimen: Demonstrasi Kelemahan Pola pada Mode ECB
Kita mengenkripsi pesan berulang (`"A"*16 + "A"*16 + "B"*16`) menggunakan ECB dan CBC, lalu membandingkan polanya.


In [ ]:
kunci_rahasia = b"KUNCI_RAHASIA123" # 16 bytes
pesan_pola = b"AAAA" * 8 + b"BBBB" * 4 # Blok 1 & 2 identik, blok 3 berbeda

# 1. Enkripsi ECB
ct_ecb = encrypt_ecb(pesan_pola, kunci_rahasia)
dec_ecb = decrypt_ecb(ct_ecb, kunci_rahasia)

# 2. Enkripsi CBC
ct_cbc, iv = encrypt_cbc(pesan_pola, kunci_rahasia)
dec_cbc = decrypt_cbc(ct_cbc, kunci_rahasia, iv)

print("=== PERBANDINGAN POLA BLOK CIPHERTEXT ===")
print("Blok Plaintext 1 (16B):", pesan_pola[0:16])
print("Blok Plaintext 2 (16B):", pesan_pola[16:32])
print("Blok Plaintext 3 (16B):", pesan_pola[32:48])

print("\n[MODE ECB] (TIDAK AMAN - Membocorkan Pola Identik):")
print("  Ciphertext Blok 1:", ct_ecb[0:16].hex())
print("  Ciphertext Blok 2:", ct_ecb[16:32].hex(), "<- PERSIS SAMA DENGAN BLOK 1!")
print("  Ciphertext Blok 3:", ct_ecb[32:48].hex())

print("\n[MODE CBC] (AMAN - Pola Tersembunyi Sempurna):")
print("  Ciphertext Blok 1:", ct_cbc[0:16].hex())
print("  Ciphertext Blok 2:", ct_cbc[16:32].hex(), "<- SANGAT BERBEDA WALAUPUN PLAINTEXT IDENTIK!")
print("  Ciphertext Blok 3:", ct_cbc[32:48].hex())


---
## 5. Eksperimen: Propagasi Error pada Mode CBC vs CTR
Apa yang terjadi jika 1 bit ciphertext rusak di tengah transmisi saluran komunikasi?


In [ ]:
pesan_panjang = b"Institut Teknologi Bandung - Program Studi Matematika - MA4151 Kriptografi Modern"

# Enkripsi CBC dan CTR
ct_cbc, iv = encrypt_cbc(pesan_panjang, kunci_rahasia)
ct_ctr, nonce = encrypt_ctr(pesan_panjang, kunci_rahasia)

# Kita korup 1 byte pada blok ke-2 (indeks byte ke-20)
ct_cbc_corrupt = bytearray(ct_cbc)
ct_cbc_corrupt[20] ^= 0x01

ct_ctr_corrupt = bytearray(ct_ctr)
ct_ctr_corrupt[20] ^= 0x01

# Dekripsi data yang terdistorsi
dec_cbc_corrupt = decrypt_cbc(bytes(ct_cbc_corrupt), kunci_rahasia, iv)
dec_ctr_corrupt = decrypt_ctr(bytes(ct_ctr_corrupt), kunci_rahasia, nonce)

print("=== ANALISIS PROPAGASI ERROR ===")
print(f"Pesan Asli: {pesan_panjang.decode()}")
print(f"\nHasil Dekripsi CBC (Error menyebar merusak 1 blok penuh + 1 bit blok berikutnya):")
print(f"  {repr(dec_cbc_corrupt)}")
print(f"\nHasil Dekripsi CTR (Hanya 1 bit di posisi yang sama persis yang salah, sisanya 100% utuh):")
print(f"  {repr(dec_ctr_corrupt)}")


---
## 6. Komparasi Komprehensif: DES vs AES-128


In [ ]:
import pandas as pd

komparasi_data = {
    "Parameter / Fitur": [
        "Tahun Diresmikan",
        "Ukuran Blok",
        "Panjang Kunci",
        "Jumlah Ronde",
        "Arsitektur Inti",
        "Fondasi Matematis",
        "Ruang Kunci",
        "Waktu Brute Force (Komputer Modern)",
        "Efisiensi Software",
        "Keamanan Saat Ini"
    ],
    "DES (Data Encryption Standard)": [
        "1977 (FIPS 46)",
        "64 bit (8 bytes)",
        "56 bit (efektif)",
        "16 ronde",
        "Feistel Network",
        "Permutasi bit & S-Box empiris",
        "2^56 ≈ 7.2 × 10^16 kombinasi",
        "Hitungan menit hingga jam",
        "Lambat (karena bit-level slicing)",
        "TIDAK AMAN (Telah dibobol)"
    ],
    "AES (Advanced Encryption Standard)": [
        "2001 (FIPS 197)",
        "128 bit (16 bytes)",
        "128, 192, 256 bit",
        "10, 12, 14 ronde",
        "Substitution-Permutation Network (SPN)",
        "Aljabar Lapangan Terhingga GF(2^8) & MDS",
        "2^128 ≈ 3.4 × 10^38 kombinasi",
        "Triliunan tahun (mustahil)",
        "Sangat cepat (byte/word 32-bit & AES-NI)",
        "STANDAR EMAS GLOBAL (Sangat Aman)"
    ]
}

df = pd.DataFrame(komparasi_data)
df


---
## 🏆 Ringkasan & Tantangan Eksperimen Mandiri
Selamat! Anda telah membangun:
1. DES murni dari permutasi bit, 8 S-box, dan 16 ronde Feistel.
2. Lapangan Galois $GF(2^8)$ dan konstruksi aljabar S-Box AES.
3. Seluruh transformasi AES-128 (`SubBytes`, `ShiftRows`, `MixColumns`, `AddRoundKey`).
4. Padding PKCS#7 dan 3 mode operasi (`ECB`, `CBC`, `CTR`).

**Tantangan untuk Anda:**
- Cobalah ubah kunci AES menjadi 192 bit atau 256 bit dengan menyesuaikan konstanta $N_k$ dan jumlah ronde ($N_r$).
- Buat simulasi enkripsi berkas teks atau citra BMP menggunakan mode CBC dan CTR!
